In [1]:
import pandas as pd
import requests
import json

from datetime import datetime
import glob
from enum import Enum
from pathlib import Path

from aind_data_schema.core.procedures import SpecimenProcedure, SpecimenProcedureType, ImmunolabelClass, HCRSeries, Antibody, Procedures, ViralMaterial

from aind_data_schema.models.organizations import Organization

from aind_data_schema.models.pid_names import PIDName

from aind_data_schema.models.registry import Registry

import logging


materials_sheet = pd.read_excel("./Mouse Tracker - RO injections.xlsx", sheet_name="Mouse Tracker - RO injections", header=[0], converters={})

c:\Users\mae.moninghoff\AppData\Local\miniconda3\envs\ainds\Lib\site-packages\openpyxl\styles\stylesheet.py:226: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [2]:
log_file_name = "./logging/log_" + datetime.now().strftime("%Y%m%d_%H%M%S") + ".log"
logger = logging.getLogger()
logger.setLevel(logging.DEBUG)
# create file handler which logs even debug messages
fh = logging.FileHandler(log_file_name, "w", "utf-8")
fh.setLevel(logging.DEBUG)

# create formatter and add it to the handlers
formatter = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")
fh.setFormatter(formatter)
# add the handlers to logger
logger.addHandler(fh)

In [ ]:
def get_inj_materials(subj_id):
    materials = []

    subj_row = materials_sheet.loc[materials_sheet["Subject ID"] == subj_id]
    for val in [1,2,3]:
        virus = subj_row[f"Virus{val}"].values[0]
        print(virus)
        if pd.isna(virus):
            continue

        virus_id = subj_row[f"Virus{val} ID" ].values[0]

        titer = subj_row[f"Virus{val} Titer (GC/mL)"].values[0]

        if pd.isna(titer):
            dose = float(subj_row[f"Virus{val} Dose (GC/mouse)"].values[0])
            volume = float(subj_row[f"Virus{val} Volume Injected"].values[0].split("u")[0])

            titer = dose/(volume*.001)


        new_material = ViralMaterial(
            name=virus,
            titer=titer,
            
        )


In [4]:
files = glob.glob("./original_spec_files/*.json")

for file in files:
    with open(file) as f:
        data = json.load(f)
        print(data)
        original_procedure = Procedures.model_construct(**data)

    print(original_procedure)

    subj = original_procedure.subject_id

    for surgery in original_procedure.subject_procedures:
        print(surgery)
        if "protocol_id" not in surgery.keys():
            surgery["protocol_id"] = "dx.doi.org/10.17504/protocols.io.kqdg392o7g25/v1"
            logging.info(f"adding surgery protocol id for subject {subj}")
        elif surgery["protocol_id"] == "unknown":
            logging.info(f"replacing surgery protocol id for subject {subj}")
            surgery["protocol_id"] = "dx.doi.org/10.17504/protocols.io.kqdg392o7g25/v1"
        for subj_procedure in surgery["procedures"]:
            if subj_procedure["procedure_type"] == "Perfusion":
                if "protocol_id" not in subj_procedure.keys():
                    logging.info(f"adding perfusion protocol id for subject {subj}")
                    subj_procedure["protocol_id"] = "dx.doi.org/10.17504/protocols.io.bg5vjy66"
                    
                elif subj_procedure["protocol_id"] == "unknown":
                    logging.info(f"replacing perfusion protocol id for subject {subj}")
                    subj_procedure["protocol_id"] = "dx.doi.org/10.17504/protocols.io.bg5vjy66"

            if subj_procedure["procedure_type"] == "Retro-orbital injection":
                
                



    # titer = dose / volume, with volume in ml (gc/ml) (translate to ml)

    # perhaps put vehicle in notes field of surgery?

    # for value in [1,2,3]:


{'describedBy': 'https://raw.githubusercontent.com/AllenNeuralDynamics/aind-data-schema/main/src/aind_data_schema/core/procedures.py', 'schema_version': '0.11.5', 'subject_id': '576404', 'subject_procedures': [{'procedure_type': 'Surgery', 'start_date': '2021-07-12', 'experimenter_full_name': '28908', 'iacuc_protocol': '1806', 'animal_weight_prior': None, 'animal_weight_post': None, 'weight_unit': 'gram', 'anaesthesia': None, 'workstation_id': None, 'procedures': [{'procedure_type': 'Perfusion', 'output_specimen_ids': ['576404']}], 'notes': None}], 'specimen_procedures': [], 'notes': None}
describedBy='https://raw.githubusercontent.com/AllenNeuralDynamics/aind-data-schema/main/src/aind_data_schema/core/procedures.py' schema_version='0.11.5' subject_id='576404' subject_procedures=[{'procedure_type': 'Surgery', 'start_date': '2021-07-12', 'experimenter_full_name': '28908', 'iacuc_protocol': '1806', 'animal_weight_prior': None, 'animal_weight_post': None, 'weight_unit': 'gram', 'anaesthes